<a href="https://colab.research.google.com/github/a-walnut-13/numerical-analysis-programming-exercises-2/blob/main/ProgrammingExercises3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Romberg Integration. Implement the Compound Trapezoidal method approxima-
tion, of I = ∫ b
a f (x) dx with Jm = 2m intervals of length hm = b−a
2m , m = 0, . . . , M . The
corresponding evaluation points are xj = a + jhm, j = 0, . . . , J. (We suppress explicit
stage mspecific notation, e.g., xj;m when the context is unambiguous.)
Then construct a Romberg table Ri,j from the resulting values to obtain much more
accurate approximations. The program structure will resemble the structure of the divided
difference table for polynomial interpolation! Analyze the error behavior of the Trapezoidal
approximations, Rm,0 = Thm , and the corresponding Romberg approximations, R0,m, with
[a, b] = [0, 1] and with f (x) given by the following functions whose integrals on [0, 1] you
can compute analytically:
i) f (x) = ex , ii) f (x) = sin4(πx) , iii) f (x) = xn, for n = 2m, 2m + 1, 2m + 2.
To obtain each successive Rm+1,0 = Thm+1 , simply form the equally weighted average of
the previous stage Trapezoidal approximation of I, Rm,0 = Thm with the approximation
obtained from equal weights of the new evaluation points at stage m + 1, are the midpoints
of the intervals of the previous stage:
xk = hm+1 + khm, k = 1, . . . , Jm, Thm+1 = Rm+1,0 = Thm + hm
Jm∑
k=1
f (xk).
(The sum on the right corresponds to the midpoint method for the intervals of the previous
stage, which is know to provide additional accuracy.) We will demonstrate why this effi-
cient formula for doubling the number of intervals the Trapezoidal reproduces the result
we would obtain if the Trapezoidal method were recomputed ‘from scratch’, ab initio .
Calculating it this way allows us to only evaluate the function f once per evaluation node!
To add the mth up and right diagonal to the Romberg table, with a similar pattern to the
divided difference table, we proceed diagonally upward to the right from row i = m − 1
column l = 1 to row i = 0 column l = m and obtain the entry from the entry immediately
below left and the entry immediately left.
Rm−l,l = c+
l Rm+1−l,l−1−c−
i Rm+l,l−1, l = 1, . . . , m, where c+
l = 4i/(4i−1), c−
l = 1/(4i−1).
Along with initialization R0,0 = (b−a)∗0.5∗(f (a)+f (b)) and hm+1 = 0.5∗hm = hm/2,
the display formulas above constitute the entire computational portion of the
program! You will discover that this is an amazingly efficient method to bootstrap pop-
ular but not very accurate approximations into spectacularly accurate approximations! It is
implemented ‘by hand’ through J = 8 = 23 here: https://www.desmos.com/calculator/mrqbjpoq9g
Romberg integration is based in part on the Euler-Maclaurin formula whose coefficients in-
volve the Bernoulli numbers we encountered in our study of power sums that also informed
our matrix algorithm complexity investigations.
https://en.wikipedia.org/wiki/Euler%E2%80%93Maclaurin%5Fformula#The%5Fformula
https://en.wikipedia.org/wiki/Euler%E2%80%93Maclaurin%5Fformula#Approximation%5Fof%5Fintegrals


In [25]:
# Programming exercises #3
import numpy as np

def romberg(f,a,b,m):
    R = np.zeros((m+1,m+1))
    h = b - a
    R[0,0] = 0.5 * h * (f(a)+f(b))

    for m in range(1,m+1):
        h = h/2
        sum_mid = 0
        num_new_pts = 2**(m-1)
        for k in range(num_new_pts):
            x = a + (2*k + 1) * h
            sum_mid += f(x)

        R[m,0] = 0.5*R[m-1,0] + h*sum_mid

    for j in range(1, m+1):
        for i in range(m-j+1):
            R[i,j] = (4**j * R[i+1,j-1] - R[i,j-1]) / (4**j - 1)

    return R

def x_squared(x): # test function f(x) = x^2
    return x*x

def exp_func(x): # test function f(x) = e^x
    return np.exp(x)

def exact_exp(a, b):
    return np.exp(b) - np.exp(a)

def sin4(x): # sin^4(x)
    return np.sin(np.pi*x)**4

def exact_sin4(a, b):
    return 3/8 * (b - a)

def power_func(n): # x^n
    return lambda x: x**n

def exact_power(n, a, b):
    return (b**(n+1) - a**(n+1)) / (n+1)


# tests

# x^2
R = romberg(x_squared, 0, 1, 4)

print("Romberg Table of x^2 from 0 to 1:")
print(R)

print("\nBest approximation:", R[0,4])
print("Exact value:", 1/3)

# e^x
R = romberg(exp_func, 0, 1, 6)

print("Romberg Table of e^x from 0 to 1:")
print(R)

print("\nBest approximation:", R[0,6])
print("Exact value:", exact_exp(0,1))

# sin^4(x)
R = romberg(sin4, 0, 1, 7)

print("Romberg Table of sin^4(x) from 0 to 1:")
print(R)

print("\nBest approximation:", R[0,7])
print("Exact value:", exact_sin4(0,1))

# x^100
R = romberg(power_func(100), 0, 1, 13)

print("Romberg Table of x^100 from 0 to 1:")
print(R)

print("\nBest approximation:", R[0,13])
print("Exact value:", exact_power(100,0,1))


Romberg Table of x^2 from 0 to 1:
[[0.5        0.33333333 0.33333333 0.33333333 0.33333333]
 [0.375      0.33333333 0.33333333 0.33333333 0.        ]
 [0.34375    0.33333333 0.33333333 0.         0.        ]
 [0.3359375  0.33333333 0.         0.         0.        ]
 [0.33398438 0.         0.         0.         0.        ]]

Best approximation: 0.3333333333333333
Exact value: 0.3333333333333333
Romberg Table of e^x from 0 to 1:
[[1.85914091 1.71886115 1.71828269 1.71828183 1.71828183 1.71828183
  1.71828183]
 [1.75393109 1.71831884 1.71828184 1.71828183 1.71828183 1.71828183
  0.        ]
 [1.7272219  1.71828415 1.71828183 1.71828183 1.71828183 0.
  0.        ]
 [1.72051859 1.71828197 1.71828183 1.71828183 0.         0.
  0.        ]
 [1.71884113 1.71828184 1.71828183 0.         0.         0.
  0.        ]
 [1.71842166 1.71828183 0.         0.         0.         0.
  0.        ]
 [1.71831679 0.         0.         0.         0.         0.
  0.        ]]

Best approximation: 1.71828182845

2. The Jacobi Method for discretized ∆u = f , u|∂C = 0 on C = [0, π] × [0, π].
We will approximating the PDE boundary problem above by Su = f . The matrix S is
symmetric and sparse. The Jacobi method can be adapted into the related Gauss-Seidel
and SOR methods. The “Multigrid Tutorial” may be helpful.
We use J for the number of interior grid points, in each direction, corresponding to J + 1
intervals with length h = π/(J + 1), and N = J2 total interior grid points. To obtain the
system Su = f , replace the second partial derivatives in the x and y directions above by
their is centered second difference approximation, incorporate the boundary conditions,
and correspond the values of the discrete approximations of u and f on the J × J 2D
grid of interior points to vectors u[n] and f [n] of length J2. Utility functions that convert
between the 1− and 2−dimensional representations using modular remainder % and integer
quotient /, are helpful: (i, j)(n) = (n%J, n/J) and n(i, j) = j ∗ J + i are inverses.
Implementation:Construct an array neighbors[n][k] that encodes the spatial relation-
ships among interior grid points. The nth row contains the indices of the free grid neigh-
bors of the nth element in the grid. an array number of neighbors[n] whose nth row
contains the number of such neighbors. To do this, use (i, j)(n) and whether i or j or
both are 0 or J − 1 to identify lower, upper, left, and right edge points and corners and
enumerate interior right-left (n ± 1) and up-down (n ± J) neighbors accordingly.
In this framework, we can compute Su in a few lines of code as:
for (n=0;n<Jsquared;n++){
sum=-4.0*u[n]; for (k=0;k<number of neighbors[n];k++) sum+=u[neighbors[n][k]];
sum/=hsquared; }
and one step of the Jacobi iteration uk+1 = Jun in a few lines of code as:
for (n=0;n<Jsquared;n++){
sum=0.0; for (k=0;k<number of neighbors[n];k++) sum+=u[neighbors[n][k]];
unew[n]=-0.25*(hsquared*f[n]-sum); }
In this form, all we have to do to implement Gauss-Seidel is replace unew[n] by u[n]!
u[n]=-0.25*(hsquared*f[n]-sum); }
A) Use the utilities to convert the 2D form of the orthogonal eigenbasis sin(lxi) sin(myj ) to
vector form, el,m[n], l, m = 1, . . . , J. Confirm orthogonality by computing dot products.
Confirm they are eigenvectors by applying S and find the eigenvalues. Apply the Jacobi
iteration to Su = el,m for some choices of l, m and analyze the behavior.
B) Convert f (xi, yj ) = (1 − ( 2
π (x − π
2 ))2)((1 − ( 2
π (y − π
2 ))2) to vector form f and solve
Su = f . Expand f in terms of the orthogonal eigenbasis using the expansion coefficient
formula. What does the Jacobi step look like in terms of the eigenbasis expansion?
C) Use the Fourier method to solve the diffusion equation initial value problem:
∂u
∂t = ∂2u
∂x2 + ∂2u
∂y2 , u(x, y, 0) = f (x, y).
All this involves is multiplying each el,m coefficient in expansion of f by e−(l2+m2)t, then
summing up the terms of the time evolved expansion!


In [ ]:
import numpy as np

def n_from_ij(i, j, J):
    return j * J + i

def ij_from_n(n, J):
    i = n % J
    j = n // J
    return i, j

def build_neighbors(J):
    N = J * J
    neighbors = [[] for _ in range(N)]

    for n in range(N):
        i, j = ij_from_n(n, J)

        if i > 0: # left
            neighbors[n].append(n_from_ij(i-1, j, J))
        if i < J-1: # right
            neighbors[n].append(n_from_ij(i+1, j, J))
        if j > 0: # down
            neighbors[n].append(n_from_ij(i, j-1, J))
        if j < J-1: # up
            neighbors[n].append(n_from_ij(i, j+1, J))

    return neighbors

def apply_S(u, neighbors, h):
    N = len(u)
    Su = np.zeros(N)

    for n in range(N):
        sum_val = -4.0 * u[n]
        for nb in neighbors[n]:
            sum_val += u[nb]
        Su[n] = sum_val / (h*h)

    return Su
